In [1]:
import os

In [2]:
%pwd

'e:\\MLProject\\MLPROJECT_WINEQUALITY\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\MLProject\\MLPROJECT_WINEQUALITY'

In [17]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
import sys

from wine_quality_prediction.constants import *
from wine_quality_prediction.utils.common import read_yaml, create_directories


In [15]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_data 
        )

        return data_ingestion_config

In [10]:
import os
import urllib.request as request
import zipfile
from wine_quality_prediction import logger
from wine_quality_prediction.utils.common import get_size

In [19]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_url,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
  

In [20]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-05 19:35:20,419 - WineQualityPrediction - INFO - yaml file: config\config.yaml loaded successfully]
[2026-03-05 19:35:20,422 - WineQualityPrediction - INFO - yaml file: params.yaml loaded successfully]
[2026-03-05 19:35:20,424 - WineQualityPrediction - INFO - yaml file: schema.yaml loaded successfully]
[2026-03-05 19:35:20,426 - WineQualityPrediction - INFO - created directory at: artifacts]
[2026-03-05 19:35:20,426 - WineQualityPrediction - INFO - created directory at: artifacts/data_ingestion]
[2026-03-05 19:35:21,688 - WineQualityPrediction - INFO - artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 